# Model Evaluation

This notebook contains the project section requested for the final ML presentation.


# Heavy Vehicle Failure Prediction Through Intelligent Sensor Data

## Complete Machine Learning Project Notebook

**Task:** Binary classification of APS-related failure in Scania heavy vehicles  
**Target:** `class` (`neg` = no APS-related failure, `pos` = APS-related failure)

This notebook contains the implementation, preprocessing, EDA, model training, tuning, evaluation, comparison, and final prediction demonstration required for the project review. Execute all cells from top to bottom before submission so the outputs and figures are visible.

The data contains 76,000 records, 170 sensor features, and a highly imbalanced target. Because missing a real failure can be costly, recall and F1-score are considered together with accuracy and ROC-AUC.


## 1. Import Libraries

The pipeline objects used later ensure that imputation and scaling are learned only from the training portion of the data. This prevents test-data leakage.


In [ ]:
# If using Google Colab and a package is missing, run this once:
# !pip install -q scikit-learn matplotlib seaborn joblib

import os
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 2. Load the Dataset

The original combined CSV is used because it contains both sensor values and the target label. The value `na` is converted to a real missing value while loading.


In [ ]:
candidate_paths = [
    Path("dataset/APS_Scania_Complete_Dataset.csv"),
    Path("APS_Scania_Complete_Dataset.csv"),
    Path("/content/APS_Scania_Complete_Dataset.csv"),
    Path("/content/HEAVY VEHICLE FAILURE PREDICTION THROUGH INTELLIGENT SENSOR/dataset/APS_Scania_Complete_Dataset.csv")
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place APS_Scania_Complete_Dataset.csv in the notebook folder or update DATA_PATH.")

df = pd.read_csv(DATA_PATH, na_values=["na"])
print("Dataset path:", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


## 4. Data Preprocessing

Duplicate rows are checked and removed before splitting. The target is encoded as 0 and 1. Missing values are not filled manually here; the model pipelines below learn training-only median values during fitting.


In [ ]:
print("Duplicate rows before removal:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Percentage", ascending=False)

print("Duplicate rows after removal:", df.duplicated().sum())
print("Columns with missing values:", int((df.isna().sum() > 0).sum()))
print("Total missing cells:", int(df.isna().sum().sum()))
display(missing_summary.head(20))


In [ ]:
df["class"] = df["class"].map({"neg": 0, "pos": 1})

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Initial dataset shape:", (76000, 171))
print("Final dataset shape after duplicate removal:", df.shape)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training class distribution:")
display(y_train.value_counts().rename_axis("class").to_frame("count"))
print("Testing class distribution:")
display(y_test.value_counts().rename_axis("class").to_frame("count"))


### Leakage Prevention

The test set is held out before model fitting. Every model below uses a pipeline. The median imputer and scaler are fitted on training data only and then applied to validation or test data. This prevents information from the test set entering the training process.


## 7. Model Building and Training

Three supervised classification models are implemented: Logistic Regression as a linear baseline, Decision Tree as an interpretable nonlinear model, and Random Forest as an ensemble model. Class weights give greater importance to the minority failure class.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=2000,
            solver="liblinear", random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            class_weight="balanced", max_depth=12,
            min_samples_leaf=5, random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
}


In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-Score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "PR-AUC": average_precision_score(y_test, probabilities)
    })
    trained_models[name] = model

comparison_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False)
display(comparison_df.round(4))
comparison_df.to_csv("model_comparison_baseline.csv", index=False)


## 8. Cross Validation

Cross-validation estimates how consistently the models perform across multiple stratified training and validation folds. The pipeline ensures that preprocessing is repeated safely inside each fold.


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}
cv_rows = []

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV F1", ascending=False)
display(cv_df.round(4))


## 9. Hyperparameter Tuning

Randomized search is used for the Random Forest. Recall is selected as the search objective because the project prioritizes detection of actual failures.


In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", "log2", 0.5]
}

search = RandomizedSearchCV(
    estimator=models["Random Forest"],
    param_distributions=param_grid,
    n_iter=10, scoring="recall", cv=3,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)
print("Best cross-validation recall:", round(search.best_score_, 4))

tuned_model = search.best_estimator_


In [ ]:
tuned_predictions = tuned_model.predict(X_test)
tuned_probabilities = tuned_model.predict_proba(X_test)[:, 1]

tuned_metrics = pd.DataFrame([{
    "Model": "Tuned Random Forest",
    "Accuracy": accuracy_score(y_test, tuned_predictions),
    "Precision": precision_score(y_test, tuned_predictions, zero_division=0),
    "Recall": recall_score(y_test, tuned_predictions, zero_division=0),
    "F1-Score": f1_score(y_test, tuned_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, tuned_probabilities),
    "PR-AUC": average_precision_score(y_test, tuned_probabilities)
}])

comparison_final = pd.concat([comparison_df, tuned_metrics], ignore_index=True)
comparison_final = comparison_final.sort_values("F1-Score", ascending=False)
display(comparison_final.round(4))
comparison_final.to_csv("model_comparison_final.csv", index=False)


## 10. Model Evaluation

The following cells provide the detailed evaluation required for the final report.


In [ ]:
print(classification_report(
    y_test, tuned_predictions,
    target_names=["No APS Failure", "APS Failure"],
    zero_division=0
))


In [ ]:
cm = confusion_matrix(y_test, tuned_predictions)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Failure", "Failure"]
)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Tuned Random Forest")
plt.show()
print("Confusion matrix values [TN, FP; FN, TP]:")
print(cm)


In [ ]:
plt.figure(figsize=(8, 6))
for name, model in trained_models.items():
    RocCurveDisplay.from_predictions(
        y_test, model.predict_proba(X_test)[:, 1], name=name
    )
RocCurveDisplay.from_predictions(
    y_test, tuned_probabilities, name="Tuned Random Forest"
)
plt.title("ROC Curves")
plt.grid()
plt.show()
